In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "WHQ_C.xpt"
csv_file = DATA_DIR / "WHQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source for fidelity comparison
source_df = df.copy()


print("=" * 78)
print("NHANES 2003-2004 WHQ_C XPT -> CSV")
print("=" * 78)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED RELEASED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WHD010",
    "WHD020",
    "WHQ030",
    "WHQ040",
    "WHD050",
    "WHQ060",
    "WHQ070",

    "WHD080A",
    "WHD080B",
    "WHD080C",
    "WHD080D",
    "WHD080E",
    "WHD080F",
    "WHD080G",
    "WHD080H",
    "WHD080I",
    "WHD080J",
    "WHD080K",
    "WHD080L",
    "WHD080M",
    "WHD080N",

    "WHQ090",

    "WHD100A",
    "WHD100B",
    "WHD100C",
    "WHD100D",
    "WHD100E",
    "WHD100F",
    "WHD100G",
    "WHD100H",
    "WHD100I",
    "WHD100J",
    "WHD100K",
    "WHD100L",
    "WHD100M",
    "WHD100N",

    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "WHQ_C columns/order differ from expected.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6213:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 42:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: WHQ_C structure = 6,213 rows x 42 columns."
)


# ============================================================
# 4. KNOWN PANDAS/XPORT TINY-ZERO ARTIFACT CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    true_zero_total
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny values found in WHQ_C. "
        "Do NOT automatically convert them to zero."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifact exists."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT:
#
# Never round genuine decimal values.
#
# Scan every numeric field BEFORE integer formatting.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine decimal values found in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values exist. "
        "They will NOT be rounded or converted:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values exist "
    "in this WHQ_C XPT."
)


# ============================================================
# 6. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):
    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
):
    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT WHOLE-NUMBER WHQ/WHD FIELDS
# ============================================================

#
# IMPORTANT:
#
# We already proved above that every released value
# in this actual WHQ XPT is whole-number.
#
# Therefore:
#
# 65.0   -> 65
# 180.0  -> 180
# 1.0    -> 1
#
# There is NO round().
#
# We also DO NOT convert/recalculate height or weight.
#

DATA_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in DATA_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Whole-number WHQ/WHD values formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC RANGE CHECK HELPER
# ============================================================

def check_range(
    col,
    range_min,
    range_max,
    range_count,
    missing_count,
    special_counts=None
):

    if special_counts is None:
        special_counts = {}

    s = df[col]


    actual_range = int(
        s.between(
            range_min,
            range_max
        ).sum()
    )

    actual_missing = int(
        s.isna().sum()
    )


    print(f"\n--- {col} ---")

    print(
        f"{range_min}-{range_max}:",
        actual_range
    )

    print(
        "Missing:",
        actual_missing
    )


    if actual_range != range_count:

        raise ValueError(
            f"{col}: CDC range count differs."
        )


    if actual_missing != missing_count:

        raise ValueError(
            f"{col}: missing count differs."
        )


    for code, expected in special_counts.items():

        actual = int(
            s.eq(code).sum()
        )

        print(
            f"{code}:",
            actual
        )


        if actual != expected:

            raise ValueError(
                f"{col}: code {code} differs."
            )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 9. CURRENT HEIGHT
# ============================================================

check_range(
    col="WHD010",
    range_min=48,
    range_max=83,
    range_count=6080,
    missing_count=52,
    special_counts={
        7777: 1,
        9999: 80
    }
)


# ============================================================
# 10. CURRENT WEIGHT
# ============================================================

check_range(
    col="WHD020",
    range_min=80,
    range_max=400,
    range_count=6116,
    missing_count=47,
    special_counts={
        7777: 2,
        9999: 48
    }
)


# ============================================================
# 11. WHQ030
# ============================================================

EXPECTED_WHQ030 = {
    1: 2971,
    2: 419,
    3: 2810,
    7: 2,
    9: 11
}


actual = (
    df["WHQ030"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_WHQ030
    or
    int(df["WHQ030"].isna().sum()) != 0
):

    raise ValueError(
        "WHQ030 differs from CDC."
    )


print(
    "\nPASS: WHQ030 matches CDC."
)


# ============================================================
# 12. WHQ040
# ============================================================

EXPECTED_WHQ040 = {
    1: 593,
    2: 3354,
    3: 2253,
    7: 1,
    9: 12
}


actual = (
    df["WHQ040"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_WHQ040
    or
    int(df["WHQ040"].isna().sum()) != 0
):

    raise ValueError(
        "WHQ040 differs from CDC."
    )


print(
    "PASS: WHQ040 matches CDC."
)


# ============================================================
# 13. WEIGHT ONE YEAR AGO
# ============================================================

check_range(
    col="WHD050",
    range_min=55,
    range_max=415,
    range_count=6121,
    missing_count=34,
    special_counts={
        7777: 2,
        9999: 56
    }
)


# ============================================================
# 14. WHQ060
# ============================================================

EXPECTED_WHQ060 = {
    1: 608,
    2: 441,
    9: 5
}


actual = (
    df["WHQ060"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_WHQ060
    or
    int(df["WHQ060"].isna().sum()) != 5159
):

    raise ValueError(
        "WHQ060 differs from CDC."
    )


print(
    "PASS: WHQ060 matches CDC."
)


# ============================================================
# 15. WHQ070
# ============================================================

EXPECTED_WHQ070 = {
    1: 1694,
    2: 3906,
    7: 1,
    9: 4
}


actual = (
    df["WHQ070"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_WHQ070
    or
    int(df["WHQ070"].isna().sum()) != 608
):

    raise ValueError(
        "WHQ070 differs from CDC."
    )


print(
    "PASS: WHQ070 matches CDC."
)


# ============================================================
# 16. WEIGHT-LOSS METHOD VARIABLES
# ============================================================

EXPECTED_WHD080 = {

    "WHD080A": {
        "counts": {
            10: 1449,
            99: 1
        },
        "missing": 4763
    },

    "WHD080B": {
        "counts": {
            11: 822
        },
        "missing": 5391
    },

    "WHD080C": {
        "counts": {
            12: 885
        },
        "missing": 5328
    },

    "WHD080D": {
        "counts": {
            13: 1345
        },
        "missing": 4868
    },

    "WHD080E": {
        "counts": {
            14: 432
        },
        "missing": 5781
    },

    "WHD080F": {
        "counts": {
            15: 249
        },
        "missing": 5964
    },

    "WHD080G": {
        "counts": {
            16: 162
        },
        "missing": 6051
    },

    "WHD080H": {
        "counts": {
            17: 148
        },
        "missing": 6065
    },

    "WHD080I": {
        "counts": {
            31: 64
        },
        "missing": 6149
    },

    "WHD080J": {
        "counts": {
            32: 157
        },
        "missing": 6056
    },

    "WHD080K": {
        "counts": {
            33: 37
        },
        "missing": 6176
    },

    "WHD080L": {
        "counts": {
            40: 65
        },
        "missing": 6148
    },

    "WHD080M": {
        "counts": {
            34: 795
        },
        "missing": 5418
    },

    "WHD080N": {
        "counts": {
            30: 233
        },
        "missing": 5980
    }
}


print("\n--- WHD080 CDC CHECKS ---")


for col, spec in EXPECTED_WHD080.items():

    actual_counts = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if (
        actual_counts != spec["counts"]
        or
        actual_missing != spec["missing"]
    ):

        raise ValueError(
            f"{col} differs from CDC.\n"
            f"Expected: {spec}\n"
            f"Actual counts: {actual_counts}\n"
            f"Actual missing: {actual_missing}"
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 17. WHQ090
# ============================================================

EXPECTED_WHQ090 = {
    1: 2230,
    2: 3978,
    7: 1,
    9: 4
}


actual = (
    df["WHQ090"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_WHQ090
    or
    int(df["WHQ090"].isna().sum()) != 0
):

    raise ValueError(
        "WHQ090 differs from CDC."
    )


print(
    "\nPASS: WHQ090 matches CDC."
)


# ============================================================
# 18. WEIGHT-GAIN-PREVENTION METHODS
# ============================================================

EXPECTED_WHD100 = {

    "WHD100A": {
        "counts": {10: 1302},
        "missing": 4911
    },

    "WHD100B": {
        "counts": {11: 692},
        "missing": 5521
    },

    "WHD100C": {
        "counts": {12: 807},
        "missing": 5406
    },

    "WHD100D": {
        "counts": {13: 1225},
        "missing": 4988
    },

    "WHD100E": {
        "counts": {14: 290},
        "missing": 5923
    },

    "WHD100F": {
        "counts": {15: 147},
        "missing": 6066
    },

    "WHD100G": {
        "counts": {16: 81},
        "missing": 6132
    },

    "WHD100H": {
        "counts": {17: 87},
        "missing": 6126
    },

    "WHD100I": {
        "counts": {31: 27},
        "missing": 6186
    },

    "WHD100J": {
        "counts": {32: 95},
        "missing": 6118
    },

    "WHD100K": {
        "counts": {33: 16},
        "missing": 6197
    },

    "WHD100L": {
        "counts": {40: 67},
        "missing": 6146
    },

    "WHD100M": {
        "counts": {34: 594},
        "missing": 5619
    },

    "WHD100N": {
        "counts": {30: 141},
        "missing": 6072
    }
}


print("\n--- WHD100 CDC CHECKS ---")


for col, spec in EXPECTED_WHD100.items():

    actual_counts = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if (
        actual_counts != spec["counts"]
        or
        actual_missing != spec["missing"]
    ):

        raise ValueError(
            f"{col} differs from CDC."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 19. WEIGHT 10 YEARS AGO
# ============================================================

check_range(
    col="WHD110",
    range_min=50,
    range_max=550,
    range_count=3525,
    missing_count=2648,
    special_counts={
        7777: 0,
        9999: 40
    }
)


# ============================================================
# 20. WEIGHT AT AGE 25
# ============================================================

check_range(
    col="WHD120",
    range_min=50,
    range_max=400,
    range_count=4227,
    missing_count=1894,
    special_counts={
        7777: 2,
        9999: 90
    }
)


# ============================================================
# 21. HEIGHT AT AGE 25
# ============================================================

check_range(
    col="WHD130",
    range_min=41,
    range_max=80,
    range_count=2415,
    missing_count=3727,
    special_counts={
        7777: 0,
        9999: 71
    }
)


# ============================================================
# 22. GREATEST WEIGHT
# ============================================================

check_range(
    col="WHD140",
    range_min=87,
    range_max=589,
    range_count=5456,
    missing_count=618,
    special_counts={
        7777: 3,
        9999: 136
    }
)


# ============================================================
# 23. AGE AT GREATEST WEIGHT
# ============================================================

check_range(
    col="WHQ150",
    range_min=11,
    range_max=84,
    range_count=5467,
    missing_count=676,
    special_counts={
        85: 44,
        77777: 0,
        99999: 26
    }
)


# ============================================================
# 24. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# WHQ_C has NO tiny-value restoration.
#
# Therefore expected values are exactly the original XPT.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


value_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:
        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 42 variables preserve "
    "the original XPT numeric values exactly."
)


# ============================================================
# 25. EXPORT CSV
# ============================================================

#
# NO float_format.
# NO recalculation.
# NO arbitrary rounding.
#
# Missing SAS numeric values become blank CSV fields.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 26. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 27. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:
        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 42 CSV columns match source XPT values."
)


# ============================================================
# 28. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 29. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 30. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 78)
print("WHQ_C CONVERSION COMPLETE")
print("=" * 78)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Tiny-value corrections: NONE")
print("Genuine fractional variables: NONE")
print("XPT -> CSV differences: 0 expected")
print("Missing-position differences: 0 expected")
print("Unnecessary .0: 0 expected")
print("Scientific notation: 0 expected")

NHANES 2003-2004 WHQ_C XPT -> CSV
Rows: 6,213
Columns: 42
PASS: WHQ_C structure = 6,213 rows x 42 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifact exists.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in this WHQ_C XPT.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31125
PASS: SEQN validated.

PASS: Whole-number WHQ/WHD values formatted without unnecessary .0.

--- WHD010 ---
48-83: 6080
Missing: 52
7777: 1
9999: 80
PASS: WHD010 matches CDC.

--- WHD020 ---
80-400: 6116
Missing: 47
7777: 2
9999: 48
PASS: WHD020 matches CDC.

PASS: WHQ030 matches CDC.
PASS: WHQ040 matches CDC.

--- WHD050 ---
55-415: 6121
Missing: 34
7777: 2
9999: 56
PASS: WHD050 matches CDC.
PASS: WHQ060 matches CDC.
PASS: WHQ070 matches CDC.

--- WHD080 CDC CHECKS ---
PASS: WHD080A
PASS: WHD080B
PASS: WHD080C
PASS: WHD080D
PASS: WHD080E
PASS: WHD080F
PASS: WHD080G
PASS: WHD080H
PASS: WHD08